# Batch norm against layer norm

A sports-watch team trains a network that reads long heart-rate recordings and flags irregular rhythm. Only two
recordings fit on its GPU per batch. In this notebook you run real `nn.BatchNorm1d` and `nn.LayerNorm` layers on
the lesson's small batches, train the lesson's tiny network with each of them, and then check a model with and
without `model.eval()`.

About 5 minutes. Run the cells from top to bottom. No GPU is needed.

## 1. Set up

The next cell imports PyTorch and NumPy and prints their versions. Colab already has both.

**Look for:** two version numbers. The lesson was run with torch 2.15.0. Another version can change the last
digit of a few numbers, but not the story.

In [1]:
import copy
import numpy as np
import torch
from torch import nn

torch.set_printoptions(precision=6)
print("torch", torch.__version__)
print("numpy", np.__version__)

torch 2.15.0
numpy 2.5.4


## 2. Batch norm by hand, on a batch of two

One number per recording goes in: its average heart rate in bpm (beats per minute). `bn_train` makes a fresh
`nn.BatchNorm1d(1)` layer, which starts in training mode, and passes the batch through it. In training mode the
layer subtracts the batch mean and divides by the batch spread. A fresh layer has weight 1 and bias 0, so you see
the plain normalized values.

**Look for:** recording A (72 bpm) comes out as -1.0 next to 148, but as +1.0 next to 50. Two close recordings,
70 and 74, still come out as about -1 and +1. Only the batch of four gives other values.

In [2]:
def bn_train(values):
    bn = nn.BatchNorm1d(1)          # training mode is the default for a fresh layer
    x = torch.tensor(values, dtype=torch.float32).reshape(-1, 1)
    return bn(x).reshape(-1).tolist()


for batch in ([72.0, 148.0], [72.0, 50.0], [70.0, 74.0], [72.0, 148.0, 90.0, 61.0]):
    out = bn_train(batch)
    print("batch", batch, "->", [round(v, 6) for v in out])

batch [72.0, 148.0] -> [-1.0, 1.0]
batch [72.0, 50.0] -> [1.0, -1.0]
batch [70.0, 74.0] -> [-0.999996, 1.0]
batch [72.0, 148.0, 90.0, 61.0] -> [-0.618731, 1.647466, -0.082001, -0.946734]


## 3. A batch of one (fails on purpose)

With one recording, the batch mean is the recording itself and the spread is 0. The cell tries it anyway. The
call is wrapped in `try/except`, so the error is printed and the notebook goes on.

**Look for:** `ValueError: Expected more than 1 value per channel when training`.

In [3]:
try:
    bn_train([72.0])
except Exception as e:  # the real error, printed as is
    print(type(e).__name__ + ":", e)

ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 1])


## 4. Layer norm ignores the batch

Recording A now has 4 hidden values: 2, 4, 6, 8. `nn.LayerNorm(4)` normalizes these 4 values of one recording.
The cell runs A alone, A with two very different partners, and A in eval mode. Then it runs batch norm on the
same 4 values, with the same two partners.

**Look for:** the four layer-norm lines are all `[-1.34164, -0.447213, 0.447213, 1.34164]`. The two batch-norm
lines differ: A's first value is -1.0 with partner 1 and 0.999995 with partner 2.

In [4]:
ln = nn.LayerNorm(4)
rec_a = torch.tensor([[2.0, 4.0, 6.0, 8.0]])
partner1 = torch.tensor([[30.0, -5.0, 0.0, 1.0]])
partner2 = torch.tensor([[0.1, 0.2, 0.1, 0.3]])

print("layer norm:")
print("  recording A alone ->", [round(v, 6) for v in ln(rec_a).reshape(-1).tolist()])
for name, p in (("with partner 1", partner1), ("with partner 2", partner2)):
    out = ln(torch.cat([rec_a, p]))[0].tolist()
    print("  recording A", name, "->", [round(v, 6) for v in out])
ln.eval()
print("  recording A, eval mode ->", [round(v, 6) for v in ln(rec_a).reshape(-1).tolist()])

print("batch norm, batch of two:")
bn4 = nn.BatchNorm1d(4)
for name, p in (("with partner 1", partner1), ("with partner 2", partner2)):
    out = bn4(torch.cat([rec_a, p]))[0].tolist()
    print("  recording A", name, "->", [round(v, 6) for v in out])

layer norm:
  recording A alone -> [-1.34164, -0.447213, 0.447213, 1.34164]
  recording A with partner 1 -> [-1.34164, -0.447213, 0.447213, 1.34164]
  recording A with partner 2 -> [-1.34164, -0.447213, 0.447213, 1.34164]
  recording A, eval mode -> [-1.34164, -0.447213, 0.447213, 1.34164]
batch norm, batch of two:
  recording A with partner 1 -> [-1.0, 1.0, 0.999999, 1.0]
  recording A with partner 2 -> [0.999995, 0.999999, 1.0, 1.0]


## 5. Running statistics and eval mode

During training, batch norm keeps a running mean and a running variance of each feature:
new = 0.9 x old + 0.1 x this batch's value. In eval mode it uses these stored values instead of the batch.
The cell passes one training batch [72, 148], then switches to eval mode.

**Look for:** the running mean moves from 0.0 to 11.0 and the running variance from 1.0 to 289.700012. In eval
mode, A alone gives 3.583898, and A inside the batch [72, 148] gives the same 3.583898. One recording now works.

In [5]:
bn = nn.BatchNorm1d(1)
print("before: running_mean", round(bn.running_mean.item(), 6), "running_var", round(bn.running_var.item(), 6), "momentum", bn.momentum)
bn(torch.tensor([[72.0], [148.0]]))           # one training-mode batch
print("after one batch [72, 148]: running_mean", round(bn.running_mean.item(), 6), "running_var", round(bn.running_var.item(), 6))
bn.eval()
print("eval mode, recording A alone [72] ->", round(bn(torch.tensor([[72.0]])).item(), 6))
print("eval mode, batch [72, 148] ->", [round(v, 6) for v in bn(torch.tensor([[72.0], [148.0]])).reshape(-1).tolist()])

before: running_mean 0.0 running_var 1.0 momentum 0.1
after one batch [72, 148]: running_mean 11.0 running_var 289.700012
eval mode, recording A alone [72] -> 3.583898
eval mode, batch [72, 148] -> [3.583898, 8.049082]


## 6. The heart-rate data and the tiny network

The next cell makes the lesson's synthetic data: 300 recordings of 240 readings (one per second), 200 for training
and 100 for validation. Label 1 means irregular (large jumps between readings), 0 means steady. Each recording has
its own resting level, so the level says nothing about the label. The seeds are fixed, so you get the lesson's data.

`make_model(norm)` is the lesson's network. One line picks the normalization: `nn.BatchNorm1d(16)` or
`nn.LayerNorm(16)`. Nothing else changes.

**Look for:** shapes `(200, 1, 240)` and `(100, 1, 240)`, and roughly half of each set labelled irregular.

In [6]:
LENGTH, N_TRAIN, N_VAL = 240, 200, 100


def make_recordings(n, rng):
    level = rng.uniform(55, 100, size=(n, 1))
    label = rng.integers(0, 2, size=n)
    jump = np.where(label[:, None] == 1, 9.0, 1.5)
    hr = level + np.cumsum(rng.normal(0, 0.3, size=(n, LENGTH)), axis=1) * 0.3 + rng.normal(0, 1, size=(n, LENGTH)) * jump
    return torch.tensor((hr - 75) / 15, dtype=torch.float32).unsqueeze(1), torch.tensor(label)


def make_model(norm):
    torch.manual_seed(0)
    norm_layer = nn.BatchNorm1d(16) if norm == "batch" else nn.LayerNorm(16)
    return nn.Sequential(
        nn.Conv1d(1, 8, kernel_size=5), nn.ReLU(),
        nn.AdaptiveAvgPool1d(1), nn.Flatten(),
        nn.Linear(8, 16), norm_layer, nn.ReLU(),
        nn.Linear(16, 2))


def accuracy(model, x, y, batch_size, order):
    correct = 0
    with torch.no_grad():
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            correct += (model(x[idx]).argmax(1) == y[idx]).sum().item()
    return correct / len(order)


rng = np.random.default_rng(0)
x_train, y_train = make_recordings(N_TRAIN, rng)
x_val, y_val = make_recordings(N_VAL, rng)
print("train:", tuple(x_train.shape), " irregular:", y_train.sum().item(), "of", N_TRAIN)
print("val:  ", tuple(x_val.shape), " irregular:", y_val.sum().item(), "of", N_VAL)

train: (200, 1, 240)  irregular: 101 of 200
val:   (100, 1, 240)  irregular: 49 of 100


## 7. Train with batches of 2: batch norm, then layer norm

`train` runs plain SGD and, after each epoch, checks validation accuracy in eval mode. Both models start from the
same seed and see the batches in the same order.

**Look for:** the batch-norm loss stays near 0.69, which is what a 50/50 guess scores on two classes, and its
validation accuracy stays near 0.5 (it ends at 0.60). The layer-norm loss falls to 0.175 by epoch 3 and 0.004 by
epoch 5, with validation accuracy 1.00 from epoch 2.

In [7]:
def train(norm, bs, epochs, report_every=1):
    model = make_model(norm)
    opt = torch.optim.SGD(model.parameters(), lr=0.05)
    loss_fn = nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(0)
    for epoch in range(1, epochs + 1):
        model.train()
        order = torch.randperm(N_TRAIN, generator=g)
        losses = []
        for i in range(0, N_TRAIN, bs):
            idx = order[i:i + bs]
            loss = loss_fn(model(x_train[idx]), y_train[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())
        model.eval()
        if epoch % report_every:
            continue
        va = accuracy(model, x_val, y_val, 2, torch.arange(N_VAL))
        print(f"{norm} norm, batch {bs:2d}, epoch {epoch}: mean train loss {sum(losses) / len(losses):.3f}  val acc (eval mode) {va:.2f}")
    return model


bn_model = train("batch", 2, 5)
ln_model = train("layer", 2, 5)

batch norm, batch  2, epoch 1: mean train loss 0.705  val acc (eval mode) 0.57
batch norm, batch  2, epoch 2: mean train loss 0.712  val acc (eval mode) 0.53


batch norm, batch  2, epoch 3: mean train loss 0.702  val acc (eval mode) 0.49


batch norm, batch  2, epoch 4: mean train loss 0.689  val acc (eval mode) 0.49
batch norm, batch  2, epoch 5: mean train loss 0.691  val acc (eval mode) 0.60


layer norm, batch  2, epoch 1: mean train loss 0.709  val acc (eval mode) 0.51
layer norm, batch  2, epoch 2: mean train loss 0.573  val acc (eval mode) 1.00


layer norm, batch  2, epoch 3: mean train loss 0.175  val acc (eval mode) 1.00
layer norm, batch  2, epoch 4: mean train loss 0.008  val acc (eval mode) 1.00


layer norm, batch  2, epoch 5: mean train loss 0.004  val acc (eval mode) 1.00


## 8. Batch norm is fine with big batches

This is "version 1" from the lesson: batch norm trained earlier on a bigger machine, with batches of 32. Batches
of 32 give only 7 steps per epoch, so it trains for 30 epochs and prints every fifth. The cell keeps a copy of the
trained weights for the next steps.

**Look for:** validation accuracy 0.75 at epoch 5 and 1.00 from epoch 10.

In [8]:
version1 = train("batch", 32, 30, report_every=5)
saved_weights = copy.deepcopy(version1.state_dict())

batch norm, batch 32, epoch 5: mean train loss 0.497  val acc (eval mode) 0.75


batch norm, batch 32, epoch 10: mean train loss 0.238  val acc (eval mode) 1.00


batch norm, batch 32, epoch 15: mean train loss 0.062  val acc (eval mode) 1.00


batch norm, batch 32, epoch 20: mean train loss 0.033  val acc (eval mode) 1.00


batch norm, batch 32, epoch 25: mean train loss 0.024  val acc (eval mode) 1.00


batch norm, batch 32, epoch 30: mean train loss 0.015  val acc (eval mode) 1.00


## 9. The check that forgot `model.eval()` (broken on purpose)

A new checking script loads version 1 and predicts in batches of 2, but it never calls `model.eval()`. It runs
under `torch.no_grad()`, which does not change how batch norm behaves. The single-recording call is wrapped in
`try/except`.

**Look for:** `model.training: True`. The accuracy changes with the shuffle (0.68, then 0.72). Recording 0 gets
P(irregular) = 0.797 with one partner and about 0.007 with the others. One recording alone raises the
`ValueError`.

In [9]:
model = make_model("batch")
model.load_state_dict(saved_weights)

print("model.training:", model.training)
for seed in (1, 2):
    order = torch.randperm(N_VAL, generator=torch.Generator().manual_seed(seed))
    print(f"val acc, batches of 2, shuffle seed {seed}: {accuracy(model, x_val, y_val, 2, order):.2f}")
with torch.no_grad():
    for partner in (1, 2, 3):
        p = model(x_val[[0, partner]]).softmax(1)[0, 1].item()
        print(f"recording 0 paired with recording {partner}: P(irregular) = {p:.3f}   (true label {y_val[0].item()})")
    try:
        model(x_val[[0]])
    except Exception as e:
        print("watch app, one recording:", type(e).__name__ + ":", e)

model.training: True
val acc, batches of 2, shuffle seed 1: 0.68
val acc, batches of 2, shuffle seed 2: 0.72
recording 0 paired with recording 1: P(irregular) = 0.797   (true label 0)
recording 0 paired with recording 2: P(irregular) = 0.007   (true label 0)
recording 0 paired with recording 3: P(irregular) = 0.008   (true label 0)
watch app, one recording: ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 16])


## 10. The fix: `model.eval()`

The cell loads a fresh copy of the same saved weights (the buggy check also changed the running statistics, more
on that in Experiment 3) and calls `model.eval()` before predicting.

**Look for:** `model.training: False`, accuracy 1.00 with both shuffles, and P(irregular) = 0.009 for recording 0
with every partner and alone.

In [10]:
model = make_model("batch")
model.load_state_dict(saved_weights)
model.eval()

print("model.training:", model.training)
for seed in (1, 2):
    order = torch.randperm(N_VAL, generator=torch.Generator().manual_seed(seed))
    print(f"val acc, batches of 2, shuffle seed {seed}: {accuracy(model, x_val, y_val, 2, order):.2f}")
with torch.no_grad():
    for partner in (1, 2, 3):
        p = model(x_val[[0, partner]]).softmax(1)[0, 1].item()
        print(f"recording 0 paired with recording {partner}: P(irregular) = {p:.3f}   (true label {y_val[0].item()})")
    p = model(x_val[[0]]).softmax(1)[0, 1].item()
    print(f"watch app, one recording: P(irregular) = {p:.3f}")

model.training: False
val acc, batches of 2, shuffle seed 1: 1.00
val acc, batches of 2, shuffle seed 2: 1.00
recording 0 paired with recording 1: P(irregular) = 0.009   (true label 0)
recording 0 paired with recording 2: P(irregular) = 0.009   (true label 0)
recording 0 paired with recording 3: P(irregular) = 0.009   (true label 0)
watch app, one recording: P(irregular) = 0.009


## Now change it

Each cell below has one line marked `# <- change this`. Edit it, run the cell again, and compare.

### Experiment 1: pick A's partner

Recording A stays at 72 bpm. Only its partner in a batch of two changes.

**What should change:** any partner above 72 gives A -1.0, any partner below 72 gives +1.0 (try 73, 1000, 71).
The size of the gap is lost. Only a partner of exactly 72 is different: both come out as 0.0.

In [11]:
partner = 148.0  # <- change this (try 73.0, 1000.0, 71.0)
print("A = 72.0, partner =", partner, "->", [round(v, 6) for v in bn_train([72.0, partner])])

A = 72.0, partner = 148.0 -> [-1.0, 1.0]


### Experiment 2: a bigger batch for batch norm

The cell trains the batch-norm model again for 5 epochs with the batch size you choose.

**What should change:** with 8, the loss falls every epoch (0.703, 0.581, 0.443, 0.307, 0.202) and
validation accuracy reaches 1.00 at epoch 5. Eight recordings give batch statistics that mean something. With 32
it learns too, but more slowly per epoch (0.75 at epoch 5), because 200 recordings make only 7 steps per epoch.

In [12]:
bs = 2  # <- change this (try 8, then 32)
_ = train("batch", bs, 5)

batch norm, batch  2, epoch 1: mean train loss 0.705  val acc (eval mode) 0.57


batch norm, batch  2, epoch 2: mean train loss 0.712  val acc (eval mode) 0.53


batch norm, batch  2, epoch 3: mean train loss 0.702  val acc (eval mode) 0.49


batch norm, batch  2, epoch 4: mean train loss 0.689  val acc (eval mode) 0.49


batch norm, batch  2, epoch 5: mean train loss 0.691  val acc (eval mode) 0.60


### Experiment 3: a check that quietly changes the model

The cell loads version 1, runs every validation recording through it in batches of 2 under `torch.no_grad()`,
and then checks it properly in eval mode. It prints the first running mean before and after, then recording 0's P(irregular)
in eval mode (0.009 in step 10).

**What should change:** with `model.train()`, the running mean moves (0.0173 becomes 0.0164) and
recording 0's P(irregular) in eval mode afterwards rises from 0.009 to 0.015. No gradient was computed and no
weight changed, yet the model is different. With `model.eval()`, the running mean stays 0.0173 and P(irregular)
stays 0.009.

In [13]:
model = make_model("batch")
model.load_state_dict(saved_weights)
model.train()  # <- change this to model.eval()
bn_layer = model[5]
print("running_mean[0] before:", round(bn_layer.running_mean[0].item(), 4))
with torch.no_grad():
    for i in range(0, N_VAL, 2):
        model(x_val[i:i + 2])
print("running_mean[0] after: ", round(bn_layer.running_mean[0].item(), 4))
model.eval()
with torch.no_grad():
    p = model(x_val[[0]]).softmax(1)[0, 1].item()
print(f"afterwards, eval mode, recording 0 alone: P(irregular) = {p:.3f}")

running_mean[0] before: 0.0173
running_mean[0] after:  0.0164
afterwards, eval mode, recording 0 alone: P(irregular) = 0.015


## What you should see

- Step 2: `nn.BatchNorm1d(1)` in training mode turns the batch [72, 148] into [-1.0, 1.0], and any batch of two
  into about -1 and +1.
- Step 3: a batch of one in training mode raises `ValueError: Expected more than 1 value per channel when training`.
- Step 5: in eval mode, A alone and A inside the batch [72, 148] both give 3.583898.
- Step 7: with batches of 2, the batch-norm model's train loss stays near 0.69 for all 5 epochs, while the
  layer-norm model's falls below 0.01 by epoch 4.
- Steps 9 and 10: without `model.eval()`, recording 0 gets a different probability with each partner; after it,
  the same probability (0.009) every time.